In [1]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)


3.12.0 2.0.2 2.3.3 1.6.1


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 2729

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)


(297, 14) 0.461 (207, 13) (90, 13)


Student ID seed: `2729`.


## Part 1

### 1. Parent entropy

$$H = -\tfrac{6}{12}\log_2\tfrac{6}{12} - \tfrac{6}{12}\log_2\tfrac{6}{12} = -0.5\cdot(-1) - 0.5\cdot(-1) = \mathbf{1.000}$$

### 2. Information gain at the root

Formula: $IG = H(\text{parent}) - \sum_k \frac{n_k}{n} H(\text{child}_k)$.

| Feature | Branch 1 (yes/no) | Branch 0 (yes/no) | Weighted entropy | Gain |
|---|---:|---:|---:|---:|
| `age_55_plus` | 6 patients: 5 / 1 | 6 patients: 1 / 5 | 0.650 | 0.350 |
| `exercise_angina` | 4 patients: 4 / 0 | 8 patients: 2 / 6 | 0.541 | 0.459 |
| `male` | 6 patients: 3 / 3 | 6 patients: 3 / 3 | 1.000 | 0.000 |
| `high_bp` | 6 patients: 4 / 2 | 6 patients: 2 / 4 | 0.918 | 0.082 |

- `age_55_plus`: $H = -\tfrac56\log_2\tfrac56 - \tfrac16\log_2\tfrac16 = 0.219 + 0.431 = 0.650$. Weighted entropy: $\tfrac{6}{12}\cdot0.650 + \tfrac{6}{12}\cdot0.650 = 0.650$. Gain: $1 - 0.650 = 0.350$.
- `exercise_angina`: branch 1 is pure, so $H=0$. Branch 0: $H = -0.25\log_2 0.25 - 0.75\log_2 0.75 = 0.5 + 0.311 = 0.811$. Weighted entropy: $\tfrac{4}{12}\cdot0 + \tfrac{8}{12}\cdot0.811 = 0.541$. Gain: $1 - 0.541 = 0.459$.
- `male`: both branches have class counts 3 / 3, so each entropy is 1. Weighted entropy: 1. Gain: 0.
- `high_bp`: each branch has proportions 2/3 and 1/3, with $H = -\tfrac23\log_2\tfrac23 - \tfrac13\log_2\tfrac13 = 0.390 + 0.528 = 0.918$. Gain: $1 - 0.918 = 0.082$.

The selected root is `exercise_angina` (gain 0.459).

### 3. Gains within the impure branch

The `exercise_angina = 1` branch is pure (4 of 4 are diseased). The impure `exercise_angina = 0` branch contains 8 patients: 2 diseased and 6 without disease, so $H = -\tfrac28\log_2\tfrac28 - \tfrac68\log_2\tfrac68 = 0.811$.

| Feature | Branch 1 (yes/no) | Branch 0 (yes/no) | Weighted entropy | Gain |
|---|---:|---:|---:|---:|
| `age_55_plus` | 2 patients: 1 / 1 | 6 patients: 1 / 5 | $\tfrac28\cdot1 + \tfrac68\cdot0.650 = 0.738$ | 0.074 |
| `male` | 3 patients: 0 / 3 | 5 patients: 2 / 3 | $\tfrac38\cdot0 + \tfrac58\cdot0.971 = 0.607$ | 0.204 |
| `high_bp` | 3 patients: 1 / 2 | 5 patients: 1 / 4 | $\tfrac38\cdot0.918 + \tfrac58\cdot0.722 = 0.796$ | 0.016 |

The selected feature is `male` (gain 0.204). It had root gain 0.000. At the root, the male/female groups have identical class proportions, so sex alone gives no gain. Within the `exercise_angina = 0` subset, the male split separates a pure group of healthy patients; the tree evaluates features within each node, allowing a feature with no root gain to be useful below the root.


## Part 2


In [3]:
def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    counts = np.bincount(y.astype(int))
    probabilities = counts[counts > 0] / len(y)
    return -np.sum(probabilities * np.log2(probabilities))

def information_gain(y, left):
    """Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    right = ~left
    return entropy(y) - left.mean() * entropy(y[left]) - right.mean() * entropy(y[right])

def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of the best split, or None if no valid split exists."""
    best = None
    best_gain = -np.inf

    for feature_index in range(X.shape[1]):
        values = np.unique(X[:, feature_index].astype(float))
        thresholds = values[:-1] + (values[1:] - values[:-1]) / 2

        for threshold in thresholds:
            left = X[:, feature_index] <= threshold
            if left.sum() < min_samples_leaf or (~left).sum() < min_samples_leaf:
                continue

            gain = information_gain(y, left)
            if gain > best_gain + 1e-12:
                best = (feature_index, float(threshold), float(gain))
                best_gain = gain

    return best


In [4]:
toy_X = np.array([
    [1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 0], [1, 1, 0, 1],
    [1, 0, 0, 1], [0, 0, 0, 0], [1, 0, 1, 0], [0, 0, 1, 0],
    [0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 0, 1], [0, 0, 0, 0],
], dtype=int)
toy_y = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0])

assert np.isclose(entropy(toy_y), 1.000)
root_gains = [information_gain(toy_y, toy_X[:, j] == 1) for j in range(4)]
assert np.all(np.isclose(np.round(root_gains, 3), [0.350, 0.459, 0.000, 0.082]))

impure_branch = toy_X[:, 1] == 0
branch_gains = [
    information_gain(toy_y[impure_branch], toy_X[impure_branch, j] == 1)
    for j in [0, 2, 3]
]
assert np.all(np.isclose(np.round(branch_gains, 3), [0.074, 0.204, 0.016]))

root_split = best_split(toy_X, toy_y)
assert root_split is not None
assert root_split[0] == 1
assert np.isclose(root_split[1], 0.5)
assert np.isclose(root_split[2], root_gains[1])

assert best_split(toy_X, toy_y, min_samples_leaf=7) is None
assert best_split(np.ones((4, 2)), np.array([0, 0, 1, 1])) is None
